# Marine Cloud Brightening TEA: interactive companion
Companion to Tito Jankowski's [mcb-tea-cost-benefit](https://github.com/titojankowski/mcb-tea-cost-benefit) workbook.
Run the cells top to bottom (Runtime > Run all). You will:
1. download Tito's workbook, 2. read the Overview, 3. change the blue input cells, 4. recalculate it in Python,
5. see the key results, 6. download your edited copy.

Nothing here changes the original on GitHub.

**Step 0. Install the tools.** `openpyxl` reads the workbook, `pycel` recalculates its formulas, `ipywidgets` gives the editable table.

In [ ]:
%pip install -q openpyxl pycel pandas ipywidgets

**Step 1. Download the workbook** from the `main` branch on GitHub (spaces in the file name are URL-encoded).

In [ ]:
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/titojankowski/mcb-tea-cost-benefit/main/"
FILE = "Draft MCB TEA and Cost Benefit TJ 10122026.xlsx"
URL = BASE + urllib.parse.quote(FILE)
XLSX = "mcb_tea_original.xlsx"
local = os.environ.get("MCB_XLSX")          # optional: use a local copy instead (for offline testing)
if local:
    import shutil; shutil.copy(local, XLSX)
else:
    urllib.request.urlretrieve(URL, XLSX)
print(URL if not local else "using local copy", "->", XLSX, os.path.getsize(XLSX), "bytes")

**Step 2. The Overview sheet** (what the workbook is for and how it is organised).

In [ ]:
import openpyxl
wb = openpyxl.load_workbook(XLSX)                    # formulas
cached = openpyxl.load_workbook(XLSX, data_only=True)  # values Excel saved last
for row in wb["Overview"].iter_rows(values_only=True):
    for v in row:
        if v: print(v if len(str(v)) < 140 else str(v)[:137] + "...")

**Step 3. Find the input cells.** The workbook's legend says *blue fill = inputs*. We detect them from the cell fill colour
on sheet `2. Cost Inputs` (a light blue, `CBE4F7`). Cells that hold a formula (for example `=D25`) are shown but locked;
only typed numbers can be edited.

In [ ]:
import pandas as pd
SHEET = "2. Cost Inputs"
ws = wb[SHEET]
def is_blue(c):
    f = c.fill
    if f is None or f.fill_type != "solid" or f.fgColor.type != "rgb": return False
    h = f.fgColor.rgb[-6:]; r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return r >= 0x90 and b > r + 20 and b >= g        # light blue; excludes grey and dark teal headers
HEAD = {r: [ws.cell(r, c).value for c in range(1, 5)] for r in range(1, ws.max_row + 1)}
def header_for(cell):                                  # nearest header row above ("Value", "Amount", ...)
    for r in range(cell.row - 1, 0, -1):
        v = ws.cell(r, cell.column).value
        if v in ("Value", "Amount", "Cost/Unit", "Selected scenario"): return v
    return ""
rows = []
for r in ws.iter_rows():
    for c in r:
        if is_blue(c) and c.value is not None:
            unit = ws.cell(c.row, 3).value if c.column != 3 else ""
            rows.append(dict(cell=c.coordinate, item=ws.cell(c.row, 1).value, field=header_for(c) or "",
                             unit=unit if isinstance(unit, str) and not unit.startswith("=") else "",
                             value=c.value, editable=not (isinstance(c.value, str) and c.value.startswith("="))))
inputs = pd.DataFrame(rows)
inputs

**Step 4. Edit the inputs.** Change any number below (the grey cells are formulas and stay locked). Nothing is recalculated until Step 5.

In [ ]:
import ipywidgets as w, json
from IPython.display import display
boxes = {}
items = []
for _, r in inputs.iterrows():
    label = w.HTML(f"<b>{r.cell}</b> {r['item']} <i>({r.field}{', ' + r.unit if r.unit else ''})</i>", layout=w.Layout(width="520px"))
    if r.editable:
        box = w.FloatText(value=float(r.value), layout=w.Layout(width="170px")); boxes[r.cell] = box
    else:
        box = w.Text(value=str(r.value), disabled=True, layout=w.Layout(width="260px"))
    items.append(w.HBox([label, box]))
# optional: edits passed as JSON in the MCB_EDITS environment variable, e.g. {"B7": 0.08} (used by the automated test)
for k, v in json.loads(os.environ.get("MCB_EDITS", "{}")).items(): boxes[k].value = v
display(w.VBox(items))

**Step 5. Recalculate in Python.** We use `pycel`, which reads each formula and evaluates it. Before applying your edits we check it
on the *unedited* workbook against the values Excel saved, and list every cell that does not agree. (We also tried the `formulas`
library: it matched too but takes about 15 s per run; `pycel` takes under 1 s.)

In [ ]:
from pycel import ExcelCompiler
import math
formula_cells = [(s.title, c.coordinate) for s in wb for r in s.iter_rows() for c in r if c.data_type == "f"]
def agree(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)): return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    return a == b
def new_engine(): return ExcelCompiler(filename=XLSX)
def check_fidelity():
    xl = new_engine(); bad = []
    for s, a in formula_cells:
        try: got = xl.evaluate(f"'{s}'!{a}")
        except Exception as e: got = f"ERROR {type(e).__name__}"
        want = cached[s][a].value
        if not agree(got, want): bad.append((s, a, wb[s][a].value, want, got))
    return len(formula_cells) - len(bad), bad
n_ok, bad = check_fidelity()
print(f"pycel reproduces {n_ok} of {len(formula_cells)} formula cells (tolerance 1e-6 relative).")
print("Cells that do NOT recalculate correctly:", "none" if not bad else "")
for b in bad: print(" ", b)

Now apply your edits and recalculate. Re-run this cell (and Step 6) after each change.

In [ ]:
KEY = {}   # label -> (sheet, cell)
S3a, S4, S6b, S6c = "3a. Outputs - Dedicated Vessel", "4. Scenario Comparison", "6b. Benefit - Global - Vessel", "6c. Benefit - Global Piggyback"
KEY["3a Levelized cost, dedicated vessel ($/t aerosol)"] = (S3a, "G5")
for lab, row in [("3a Dedicated vessel", 6), ("3b Ship piggyback", 7), ("3c Fixed platform", 8), ("3d Stationary vessel", 9)]:
    KEY[f"4 {lab} ($/t)"] = (S4, f"B{row}")
for lab, row in [("Low", 20), ("Central", 21), ("High", 22)]:
    KEY[f"6b value per degree-year, {lab} ($B/K-yr)"] = (S6b, f"C{row}")
for lab, row in [("Low", 29), ("Central", 30), ("High", 31)]:
    KEY[f"6c value per degree-year, {lab} ($B/K-yr)"] = (S6c, f"C{row}")
for lab, row in zip(["CESM2", "CESM2-Ghosh", "E3SMv2", "UKESM1.0"], range(10, 14)):
    KEY[f"6b benefit-cost, {lab}"] = (S6b, f"D{row + 19}")
for lab, row in zip(["CESM2", "CESM2-Ghosh", "E3SMv2", "UKESM1.0"], range(38, 42)):
    KEY[f"6c benefit-cost, {lab}"] = (S6c, f"D{row}")
edits = {a: b.value for a, b in boxes.items() if b.value != ws[a].value}
xl = new_engine()
for s, a in KEY.values(): xl.evaluate(f"'{s}'!{a}")   # pycel must build the dependency graph before inputs can change
for a, v in edits.items(): xl.set_value(f"'{SHEET}'!{a}", v)
table = pd.DataFrame([dict(output=k, workbook_cached=cached[s][a].value, recalculated=xl.evaluate(f"'{s}'!{a}"), cell=f"{s.split('.')[0]}!{a}")
                      for k, (s, a) in KEY.items()])
table["change_%"] = (table.recalculated / table.workbook_cached - 1) * 100
print("Edited inputs:", edits or "none (showing the unedited workbook)")

**Step 6. Key outputs.** `workbook_cached` is what Tito's file shows; `recalculated` includes your edits.
Value per degree-year = global GDP x damage % per degC, in $ billion per K-year (sheets 6b rows 20-22 and 6c rows 29-31).
Benefit-cost ratios are on 6b D29:D32 and 6c D38:D41.

In [ ]:
pd.options.display.float_format = "{:,.4g}".format
table

**Step 7. Download your edited workbook.** Your input changes are written into a copy of the original (formulas kept, Excel recalculates when you open it).

In [ ]:
OUT = "mcb_tea_edited.xlsx"
out = openpyxl.load_workbook(XLSX)
for a, v in edits.items(): out[SHEET][a].value = v
out.calculation.fullCalcOnLoad = True
out.save(OUT)
try:
    from google.colab import files
    files.download(OUT)
except ImportError:                      # plain Jupyter: show a link instead
    from IPython.display import FileLink, display
    display(FileLink(OUT))